In [1]:
# Work around a TRL 0.24.0 + Transformers 5.5.0 bug where prompt-only apply_chat_template() returns a BatchEncoding, causing TRL to miscompute the prompt length (e.g., len(prompt_ids) == 2) and incorrectly mask only the first few prompt tokens.
# SFT caompatibe != TRL 0.24.0 & Transformers 5.5 -> prompt-only apply_chat_template() error reported
# %pip install --upgrade --no-cache-dir transformers trl datasets peft accelerate bitsandbytes safetensors
# !pip install --upgrade "torchao>0.16.0" # PEFT requirement

In [2]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Colab_Notebooks') # change directory to the current working directory

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import pandas as pd
from datasets import Dataset

DPO_TRAIN_DATA_LOAD_PATH = "data_preference_train_train_only/bridge_2tage__with_reference_solution_more_models/response_pairs_df.csv"
TEST_DATA_LOAD_PATH = "train_test_split/test_stepverify_labeled_0.9.json"

BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

SFT_ADAPTER_PATH = "llama3-8b-instruct-sft-adapter" # Adapter name # DPO (pi_theat, pi_ref)

DPO_ADAPTER_PATH_05 = "llama3-8b-instruct-dpo-adapter--threshold_05" # DPO_ADAPTER_SAVE_PATH
DPO_ADAPTER_PATH_03 = "llama3-8b-instruct-dpo-adapter--threshold_03" # DPO_ADAPTER_SAVE_PATH
DPO_ADAPTER_PATH_01 = "llama3-8b-instruct-dpo-adapter--threshold_01" # DPO_ADAPTER_SAVE_PATH

DRIVE_ROOT_DIR = "/content/drive/My Drive/Colab_Notebooks" if IN_COLAB else "" # current notebook directory in the google drive

DPO_MODEL_DIR_05 = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_PATH_05)    # save in the ADAPTER directory in the google drive  # DPO_DRIVE_MODEL_DIR
DPO_MODEL_DIR_03 = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_PATH_03)    # save in the ADAPTER directory in the google drive  # DPO_DRIVE_MODEL_DIR
DPO_MODEL_DIR_01 = os.path.join(DRIVE_ROOT_DIR, DPO_ADAPTER_PATH_01)    # save in the ADAPTER directory in the google drive  # DPO_DRIVE_MODEL_DIR

SFT_DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT_DIR, SFT_ADAPTER_PATH)

SEED = 42

In [4]:
# from huggingface_hub import notebook_login
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
login(token=hf_token )

#### Load and format dataset

In [5]:
SYSTEM_TEMPLATE = """You are an experienced elementary mathematics tutor. Your role is not merely to correct the student's mistake. Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson. Your task is to guide the student's response to the problem.

Your response should be at most two sentences.
"""

USER_TEMPLATE = """
### Problem:
{problem}

### Student's response:
{student}
"""


def make_dpo_test_dataset(data):
    dataset = Dataset.from_list([
        {
            # Metadata
            "topic": row["topic"],
            "problem": row["problem"],
            "student_mistake": row["student_mistake"],
            "reference_solution": row["reference_solution"],
            "error_category": row["error_category"],
            "error_description": row["error_description"],
            "student_correct_response": row["student_correct_response"],

            "prompt": [
                {
                    "role": "system",
                    "content": SYSTEM_TEMPLATE,
                },
                {
                    "role": "user",
                    "content": USER_TEMPLATE.format(
                        problem=str(row["problem"]),
                        student=str(row["student_mistake"]),
                    ),
                },
            ],
            "completion": [
              {
                  "role": "assistant",
                  "content": "(" + str(row['dialog_history'][0]['pedagogy']) + ")" + str(row['dialog_history'][0]['text']),
              }
          ],
        }
        for row in data
    ])

    return dataset

In [6]:
test_data = json.load(open(TEST_DATA_LOAD_PATH, "r"))
# test_data = pd.DataFrame(test_data).rename(columns={"student_incorrect_solution": "student_mistake"}).to_dict(orient="records") # change the key name to match pref_data_format
for row in test_data:
    row["student_mistake"] = row.pop("student_incorrect_solution")
test_ds = make_dpo_test_dataset(test_data)

#### Generate responses per model 
- the base model is initially loaded and then each adapter is alternatively attached to the base model.

In [7]:
############
import torch
from transformers import pipeline
from tqdm.auto import tqdm
from copy import deepcopy
from transformers.utils import logging

logging.set_verbosity_error() # turn off warning ( both max_new_tokens and max_length are set .... )

def get_base_model_and_tokenizer():

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        device_map={"": 0},
    )

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        clean_up_tokenization_spaces=False,
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()

    return model, tokenizer


def check_adapter(model):
    print(f"Checking adapter status..")
    if hasattr(model, "peft_config") and model.peft_config:
        print(f"  Loaded adapters ( {list(model.peft_config.keys())} )")
        print(f"  Active adapter ( {model.active_adapters()} )")
        assert model.active_adapters(), "Adapter is loaded but no adapter is active."
    else:
        print(f"  No adapter loaded")



def generate_responses(model, tokenizer, dataset, batch_size=64):

    check_adapter(model) # check if model has an adapter

    prompts = [
        tokenizer.apply_chat_template(
            data["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )
        for data in dataset
    ]

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text=False,
    )

    outputs = []

    print("Generating responses..")
    print(
        f"  Total Rows: {len(prompts)} | "
        f"Batch Size: {batch_size} | "
        f"Total Batches: {(len(prompts) + batch_size - 1) // batch_size} |\n"
    )

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_outputs = pipe(
            prompts[i:i + batch_size],
            do_sample=False,
            clean_up_tokenization_spaces=True,
        )

        outputs.extend(batch_outputs)

    results = {
        "student_mistake": [ row["student_mistake"] for row in dataset ],
        "topic": [ row["topic"] for row in dataset ],
        "problem": [ row["problem"] for row in dataset ],
        "error_category": [ row["error_category"] for row in dataset ],
        "error_description": [ row["error_description"] for row in dataset ],
        "reference_solution": [ row["reference_solution"] for row in dataset ],
        "student_correct_response": [ row["student_correct_response"] for row in dataset ],
        "ground_truth": [ row["completion"][0]["content"] for row in dataset ],
        "prompts": prompts,
        "llm_response": [ output[0]["generated_text"].strip() for output in outputs ],

    }

    print("=" * 100)

    return results



def generate_all_responses(dataset, adapter_dirs):
    base_model, base_tokenizer = get_base_model_and_tokenizer()

    results = {}

    # Base model
    print("[ Current Model : base ]")
    results["base"] = generate_responses( base_model, base_tokenizer, dataset)

    # Fine-tuned models
    for model_name, adapter_dir in adapter_dirs.items():
        print(f"[ Current Model : {model_name} ]")

        # Load adapter using the same temporary name
        base_model.load_adapter(adapter_dir, is_trainable=False )

        base_model.set_adapter("default") # activate the adapter. default name = "default" https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/peft?utm_source=chatgpt.com

        # Generate
        results[model_name] = generate_responses(base_model, base_tokenizer, dataset,)

        # Remove adapter before loading the next one
        base_model.delete_adapter("default")

    return results

W0902 02:45:41.827000 85842 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0902 02:45:41.881000 85842 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0902 02:45:41.908000 85842 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [ ]:
adapter_dirs = {
    "sft": SFT_DRIVE_MODEL_DIR,
    "dpo_01": DPO_MODEL_DIR_01,
    "dpo_03": DPO_MODEL_DIR_03,
    "dpo_05": DPO_MODEL_DIR_05,
}

model_responses = generate_all_responses(test_ds, adapter_dirs,)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

#### Save generated responses

In [ ]:
model_responses = json.load(open("model_responses_sft_dpo__raw.json", "r"))

In [ ]:
import json

with open("model_responses_sft_dpo__raw.json", "w", encoding="utf-8") as f:
    json.dump(model_responses, f, ensure_ascii=False, indent=2, )

#### Sort and reconstruct dataset 

In [ ]:
# common data
topics = model_responses['base']['topic']
problems = model_responses['base']['problem']
error_categories = model_responses['base']['error_category']
error_descriptions = model_responses['base']['error_description']
reference_solutions = model_responses['base']['reference_solution']
student_correct_responses = model_responses['base']['student_correct_response']
student_mistakes = model_responses['base']['student_mistake']
ground_truth = model_responses['base']['ground_truth']
prompts = model_responses['base']['prompts']

# responses
base_responses = model_responses['base']['llm_response']
sft_responses = model_responses['sft']['llm_response']
dpo_01_responses = model_responses['dpo_01']['llm_response']
dpo_03_responses = model_responses['dpo_03']['llm_response']
dpo_05_responses = model_responses['dpo_05']['llm_response']

model_responses__sorted__df = pd.DataFrame({
    "topic": topics,
    "problem": problems,
    "error_category": error_categories,
    "error_description": error_descriptions,
    "reference_solution": reference_solutions,
    "student_correct_response": student_correct_responses,
    "student_mistake": student_mistakes,
    "original_tutor_response": ground_truth,
    "prompt": prompts,

    "base": base_responses,
    "sft": sft_responses,
    "dpo_01": dpo_01_responses,
    "dpo_03": dpo_03_responses,
    "dpo_05": dpo_05_responses,
})

model_responses__sorted__df.to_csv("model_responses_sft_dpo__sorted_df.csv")

#### Print Prompt and Responses per model 

In [ ]:
import pandas as pd
model_responses__sorted__df = pd.read_csv("model_responses_sft_dpo__sorted_df.csv")

In [ ]:
for idx, row in model_responses__sorted__df.iterrows():

    print(f"Index: {idx}")

    print(f"prompt: {row['prompt']}")
    print("="*100)
    for column, value in row.items():
        if column == "sft":
            print("SFT")
            print(value)
        elif column == "dpo_01":
            print("DPO_01")
            print(value)
        elif column == "dpo_03":
            print("DPO_03")
            print(value)
        elif column == "dpo_05":
            print("DPO_05")
            print(value)


    print("=" * 100)

prompt
: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an experienced elementary mathematics tutor.
Your role is not merely to correct the student's mistake.
Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson.
Your task is to guide the student's response to the problem.

Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Problem:
lori owns a carsharing company.
there are three red cars and two white cars available to rent.
renting the white car costs 2 for every minute and the red car 3 for every minute.
all cars were rented for 3 hours.
how much money did lori earn?

### Student's response:
'renting the white car for 3 hours 180 minutes costs 2 x 180 360.', 'renting the red car for 3 hours 180 minutes costs 3 x 180 540 per car.', 'lori earned 3 x 540 1620 for the three red cars.', 'lori earn

In [ ]:
# print("""
# prompt
# : <|begin_of_text|><|start_header_id|>system<|end_header_id|>

# You are an experienced elementary mathematics tutor.
# Your role is not merely to correct the student's mistake.
# Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
# You will be given a problem from an elementary mathematics lesson.
# Your task is to guide the student's response to the problem.

# Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

# ### Problem:
# this weekend's football game matched the seattle seahawks with the denver broncos.
# the final score was 37-23, with the seattle seahawks taking the win.
# if a touchdown is worth 7 points and a field goal is worth 3 points, how many touchdowns did the seattle seahawks score if they scored 3 field goals during the game?

# ### Student's response:
# 'the seattle seahawks scored 23 points in total, and they scored 3 field goals, which is 3 x 3 9 points from field goals.', 
# 'to find out how many points they scored from touchdowns, we can subtract the points from field goals from the total points 23 - 9 14.', 'since a touchdown is worth 7 points, the seattle seahawks scored 14 7 2 touchdowns.', 
# ' 2'<|eot_id|><|start_header_id|>assistant<|end_header_id|>


# error category: 
# missing_wrong_factual_knowledge
# error description: 
# The student needed to subtract the points from field goals from the TOTAL Seahawk points, which was 37, not 23
# ====================================================================================================
# original tutor response
# (generic)hi cody, can you tell me how you got your answer?
# SFT
# how many points did the seahawks score in total?
# DPO_01
# no, the seattle seahawks scored 37 points in total.
# DPO_03
# no, you have made a mistake. the seattle seahawks scored 37 points in total.
# DPO_05
# no, you have already subtracted the points from field goals from the total points. you need to add the points from field goals to the points from touchdowns.
# ====================================================================================================
# """)


prompt
: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an experienced elementary mathematics tutor.
Your role is not merely to correct the student's mistake.
Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson.
Your task is to guide the student's response to the problem.

Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Problem:
this weekend's football game matched the seattle seahawks with the denver broncos.
the final score was 37-23, with the seattle seahawks taking the win.
if a touchdown is worth 7 points and a field goal is worth 3 points, how many touchdowns did the seattle seahawks score if they scored 3 field goals during the game?

### Student's response:
'the seattle seahawks scored 23 points in total, and they scored 3 field goals, which is 3 x 3 9 points from field goals.', 
't

In [ ]:
print("""
prompt
: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an experienced elementary mathematics tutor.
Your role is not merely to correct the student's mistake.
Your broader goal is to help the student remain curious, confident, and capable of thinking for themselves.
You will be given a problem from an elementary mathematics lesson.
Your task is to guide the student's response to the problem.

Your response should be at most two sentences.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Problem:
forty percent of the students have elected to learn from home during the pandemic.
the remaining students are divided into two equal groups, only one of which is physically in school on any day.
what percent of students are present in school?

### Student's response:
'if 40 of the students are learning from home, then 60 are in school 100 - 40 60 .
', 'the remaining 60 of students are divided into two equal groups, so each group represents 30 of the total students 60 2 30 .
', 'since only one of the two groups is physically in school on any day, that means 30 2 15 of the total students are present in school on any given day.
', ' 15'<|eot_id|><|start_header_id|>assistant<|end_header_id|>


error category: reached_correct_solution_but_proceeded_further
====================================================================================================
original tutor response
(generic)hi deandre, could you please explain your approach to this problem?
SFT
if 40 of the students are learning from home, then 60 are in school 100 - 40 60 . the remaining 60 of students are divided into two equal groups, so each group represents 30 of the total students 60 2 30 . 
since only one of the two groups is physically in school on any day, that means 30 2 15 of the total students are present in school on any given day.
DPO_01
if 40 of the students are learning from home, then 60 are in school 100 - 40 60 . the remaining 60 of students are divided into two equal groups, so each group represents 30 of the total students 60 2 30 . 
since only one of the two groups is physically in school on any day, that means 30 2 15 of the total students are present in school on any given day. 15
DPO_03
you have made a mistake. the remaining 60 of students are divided into two equal groups, so each group represents 30 of the total students 60 2 30 .
DPO_05
you have already told me that 60 students are in school, so you don't need to do any more calculations.
""")

In [ ]:
model_responses__sorted__df

,Unnamed: 0,topic,error_category,error_description,reference_solution,student_correct_response,stepverify_original,prompt,base,sft,dpo_01,dpo_03,dpo_05
0,0,Math Word Problem,misunderstanding_of_a_question,NaN,Both white cars were rented for 2 * 2 = $4 for...,Lori had 3 red cars and 2 white cars available...,"(generic)hi brenda, could you please walk me t...",<|begin_of_text|><|start_header_id|>system<|en...,That's a great start! You're correct that the ...,how many cars are there in total?,"no, you have already calculated the cost of th...",great!,"no, you have already multiplied by 3 for the r..."
1,1,Math Word Problem,extra_quantity_or_missing_quantity,Added quantity,"At 7 days a week, it takes Bert 2 * 7 = 14 day...","Bert uses up a pencil every two weeks, which i...","(generic)hi scott, could you please walk me th...",<|begin_of_text|><|start_header_id|>system<|en...,I love the student's enthusiasm and effort! Ho...,if it takes 1050 words to use up 1 14 of a pen...,"if bert uses up a pencil every two weeks, then...","no, if he uses up a pencil every two weeks, th...","no, if he uses up a pencil every two weeks, th..."
2,2,Math Word Problem,misunderstanding_of_a_question,NaN,Since he fires his weapon on average every 15 ...,To find out how many times Jason fires his wea...,"(generic)hi , could you please walk me through...",<|begin_of_text|><|start_header_id|>system<|en...,I love the student's creative approach! Howeve...,how many times does jason shoot flames in a mi...,"no, you are correct that jason shoots flames f...",let's try again. jason fires his weapon every ...,"no, you have made a mistake. you have multipli..."
3,3,Math Word Problem,none_of_the_above,student calculated the value of a variable h a...,The weekly salary of Agnes is $15 x 8 = $120.\...,To find out how many hours Mila needs to work ...,"(generic)hi , could you please walk me through...",<|begin_of_text|><|start_header_id|>system<|en...,I love how you broke down the problem step by ...,how many hours does mila need to work to earn ...,"no, you have already solved the problem. you h...",you have already worked out that agnes earns 4...,you have already worked out that agnes earns 4...
4,4,Math Word Problem,misunderstanding_of_a_question,NaN,Jenny spent 1 - 3/7 = 4/7 of her money.\nSince...,"Jenny spent 3/7 of her money, which means she ...",(generic)hi can you talk me through your answer,<|begin_of_text|><|start_header_id|>system<|en...,I love the effort you put into solving the pro...,jenny spent 3 7 of her money. what is 3 7 of 84?,"no, the correct answer is 56.",let's try again. let's say her original amount...,"no, you have made a mistake. the equation is 4..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,293,Math Word Problem,missing_wrong_factual_knowledge,The student needed to subtract the points from...,The Seattle Seahawks scored 3 field goals duri...,The Seattle Seahawks scored 3 field goals duri...,"(generic)hi cody, can you tell me how you got ...",<|begin_of_text|><|start_header_id|>system<|en...,I love the student's attempt to break down the...,how many points did the seahawks score in total?,"no, the seattle seahawks scored 37 points in t...","no, you have made a mistake. the seattle seaha...","no, you have already subtracted the points fro..."
294,294,Math Word Problem,none_of_the_above,method of approaching problem,He made $150 selling the cards because 100 + 5...,Kenny wanted to buy a $100 ticket to the amuse...,"(generic)hi, can you talk me through your solu...",<|begin_of_text|><|start_header_id|>system<|en...,I love the effort! You're absolutely on the ri...,let's look at the first part of the question. ...,let x be the total number of cards kenny start...,let's try again. let's say you have 12 cookies...,you have already said he has 50 in spending ca...
295,295,Math Word Problem,extra_quantity_or_missing_quantity,Student works out the cost of 24 instead of 14,"To buy 14 balloons, Harry should buy one pack ...","To bu